# Question c)
Finite diff method



In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm



## Explicit
Did not use implicit or Crank Nielson in the end


the first derivative wrt x is given by (using central)
-     $\frac{1}{2}(v_{j+1}-v_{j-1})$

the second derivative (using central)
-     $v_{j-1}-2v_{j}+v_{j+1}$

We have found the following PDE
- $ V_t + \frac{1}{2}\sigma^2(S_1^2*V_{11} +S_2^2*V_{22}) = r(V - V_1S_1-V_2S_2)$


Looking only at the change caused by one of the stocks (lets say $S_1$)
- $\frac{1}{2}\sigma^2(S_1^2*V_{11}) = rV_1S_1$


But now as a function of $V_x$ and $V_{xx}$

- $dx = \frac{1}{S_1} dS$

- $\frac{1}{2}\sigma^2(V_{xx} - V_x)+ r*V_x$

In [2]:
# first derivative wrt x= log(S/S0) using central method
def get_d1(n):
    return 0.5*(np.diag(np.ones(n-1), 1)-np.diag(np.ones(n-1), -1))
def get_d2(n):
    return np.diag(np.ones(n-1), 1)+np.diag(-2.*np.ones(n))+np.diag(np.ones(n-1), -1)
def get_l(n, dx, r, sig):
    return (r-0.5*sig**2.)/dx * get_d1(n) + 0.5*sig**2./dx**2. * get_d2(n)

In [3]:
def analytic_value(S1, S2, sigma, T):
    sig_hat = sigma*np.sqrt(2.)
    d1 = (np.log(S1/S2)+0.5*sig_hat**2.*T)/(sig_hat*np.sqrt(T))
    d2 = d1-sig_hat*np.sqrt(T)
    return S2+S1*norm.cdf(d1)-S2*norm.cdf(d2)

In [4]:
def max_boundary(x, S0):
    S1, S2 = np.meshgrid(S0*np.exp(x), S0*np.exp(x), indexing='ij')
    return np.maximum(S1, S2)

def create_and_run_explicit_max(n_x, n_t, x_lim, r, sigma, T, S0):

    assert n_x % 2 == 1, f"The provided n_x={n_x} is even, (S0, S0) would not be a grid point."
    dx = 2.*x_lim/(n_x-1)
    dt = T/(n_t-1)

    x = np.linspace(-x_lim, x_lim, n_x)
    explicit_grid = max_boundary(x, S0)
    matrix = get_l(n_x, dx, r, sigma)

    for n in range(1, n_t)[::-1]:
        change = np.dot(matrix, explicit_grid) + np.dot(explicit_grid, matrix.T) - r*explicit_grid
        #Drop the boundary grid
        explicit_grid[1:-1,1:-1] = (explicit_grid + dt*change)[1:-1,1:-1]

    return explicit_grid

In [5]:
r, sigma, T, S0 = 0.05, 0.2, 1.0, 100.0

n_x, n_t, x_lim = 201, 713, 1.5

explicit_grid = create_and_run_explicit_max(n_x, n_t, x_lim, r, sigma, T, S0)
V_pde = explicit_grid[n_x//2, n_x//2]
V_analytic = analytic_value(S0, S0, sigma, T)

dx = 2.*x_lim/(n_x-1)
dt = T/(n_t-1)
print(f"PDE:      {V_pde:.6f}")
print(f"Analytic: {V_analytic:.6f}")
print(f"Error:    {V_pde-V_analytic:+.6f}")

PDE:      111.245187
Analytic: 111.246292
Error:    -0.001105


In [6]:
# The fair value does not depend on r, so the PDE value should barely move
print("Dependence on r:")
for test_r in [0., 0.02, 0.05, 0.1]:
    test_grid = create_and_run_explicit_max(n_x, n_t, x_lim, test_r, sigma, T, S0)
    print(f"  r={test_r:<5} PDE={test_grid[n_x//2, n_x//2]:.6f}")

print("Dependence on the grid width:")
fixed_dx, fixed_n_t = 0.02, 401
for test_lim in [0.5, 1., 1.5, 2., 3.]:
    test_n_x = int(round(2.*test_lim/fixed_dx))+1
    test_grid = create_and_run_explicit_max(test_n_x, fixed_n_t, test_lim, r, sigma, T, S0)
    print(f"  x_lim={test_lim:<4} error={test_grid[test_n_x//2, test_n_x//2]-V_analytic:+.6f}")

Dependence on r:
  r=0.0   PDE=111.244192
  r=0.02  PDE=111.244590
  r=0.05  PDE=111.245187
  r=0.1   PDE=111.246181
Dependence on the grid width:
  x_lim=0.5  error=-0.006106
  x_lim=1.0  error=-0.001959
  x_lim=1.5  error=-0.001959
  x_lim=2.0  error=-0.001959
  x_lim=3.0  error=-0.001959


# Question d)

In [7]:
#this one came from AI 
def monte_carlo_value(n_paths, r, sigma, T, S0, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, 2)) #because n senarios for 2 differnt stocks
    S_T = S0*np.exp((r-0.5*sigma**2.)*T + sigma*np.sqrt(T)*z)
    discounted_payoff = np.exp(-r*T)*S_T.max(axis=1) # the max of the two columns (stocks) is taken in each row 
    return discounted_payoff.mean(), discounted_payoff.std(ddof=1)/np.sqrt(n_paths)

In [8]:
n_paths = 10**7
V_mc, mc_error = monte_carlo_value(n_paths, r, sigma, T, S0)

print(f"Analytic:    {V_analytic:.6f}")
print(f"PDE:         {V_pde:.6f}   error wrt analytic: {V_pde-V_analytic:+.6f}")
print(f"Monte Carlo: {V_mc:.6f}   error wrt analytic: {V_mc-V_analytic:+.6f}   std error: {mc_error:.6f}")


Analytic:    111.246292
PDE:         111.245187   error wrt analytic: -0.001105
Monte Carlo: 111.241942   error wrt analytic: -0.004350   std error: 0.005917


expectation: the error of monte carlo should $~ \frac{1}{\sqrt{n}}$
So 10x more paths should give error of approx 3.1 smaller. 

In [9]:
print(" n_paths   std error")
for test_paths in [10**3, 10**4, 10**5, 10**6, 10**7]:
    test_value, test_error = monte_carlo_value(test_paths, r, sigma, T, S0)
    print(f"{test_paths:8d}  {test_error:10.6f} ")

 n_paths   std error
    1000    0.575449 
   10000    0.185012 
  100000    0.059108 
 1000000    0.018715 
10000000    0.005917 


## conclusions
PDE is more accurate in our experiment.  

But pde is computationally more expensive as we consider more stocks. Monte carlo computation stays controlled: for each stock added we would need one extra random number for the monte carlo method.(one extra column) Whereas for PDE we need to calculate an entery on the grid for each combination of stock prices. 


